## Результаты алгоритма без реранкера и дообучения

In [1]:
import polars as pl

In [2]:
golden_set = pl.read_parquet("../data/golden_set.parquet").filter(pl.col("geonameIds").list.len() > 0)

In [3]:
import httpx
from tqdm.auto import tqdm


def search_batch(queries: list[str], top_k: int = 50, use_rerank: bool = True):
    base_url = "http://localhost:8000/v1/search"
    results = []
    
    with httpx.Client(timeout=30.0) as client:  # синхронный клиент
        for query in tqdm(queries):
            response = client.get(base_url, params={"text": query, "top_k": top_k, "use_rerank": use_rerank})
            response.raise_for_status()
            results.append(response.json())
    
    return results

In [4]:
from ir_measures import P, Recall, RR, calc


qrels_dict = {}
for row in golden_set.iter_rows(named=True):
    qrels_dict.update({row["query"]: {str(gid): 1 for gid in row["geonameIds"]}})

predictions = search_batch(qrels_dict.keys(), top_k=50, use_rerank=False)

  0%|          | 0/180 [00:00<?, ?it/s]

In [5]:
run_dict = {
    pred["query"]: {
        str(r["geonameid"]): len(pred["results"]) - i
        for i, r in enumerate(pred["results"])
    }
    for pred in predictions
}

metrics = calc([RR, P@1, Recall@5, Recall@10, Recall@25, Recall@50], qrels_dict, run_dict)
metrics_aggregated = pl.DataFrame({str(k): v for k, v in metrics.aggregated.items()}).unpivot().sort("value")

metrics_per_query = {str(m): {"query": [], "value": []} for m in metrics.aggregated}

for metric in metrics.per_query:
    mname = str(metric.measure)
    metrics_per_query[mname]["query"].append(metric.query_id)
    metrics_per_query[mname]["value"].append(metric.value)

for mname in metrics_per_query:
    metrics_per_query[mname] = pl.DataFrame(metrics_per_query[mname]).sort("value")

In [6]:
metrics_aggregated

variable,value
str,f64
"""P@1""",0.838889
"""R@5""",0.840895
"""RR""",0.87488
"""R@10""",0.889654
"""R@25""",0.943873
"""R@50""",0.961524


In [7]:
for k in metrics_per_query:
    print(k)
    display(metrics_per_query[k].limit(5))

R@50


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Is Omsk or Yaroslavl a better …",0.5
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""Какая погода в Ване в конце ок…",0.5


P@1


query,value
str,f64
"""Tucson, Arizona'da yaz mevsimi…",0.0
"""What are some hidden gems to v…",0.0
"""Amerika'da kariyer fırsatları …",0.0
"""Какая погода ожидается в Чите …",0.0
"""Как добраться из центра до жел…",0.0


R@25


...

query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Van Gölü etrafında sessiz bir …",0.0
"""Какие достопримечательности ст…",0.0
"""Где лучше жить в США для споко…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0


R@10


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Van Gölü etrafında sessiz bir …",0.0
"""Какие достопримечательности ст…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0
"""Van Gölü'nün kenarında doğal g…",0.0


RR


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Van Gölü etrafında sessiz bir …",0.04
"""Van Gölü'nün kenarında doğal g…",0.04
"""Van Gölü'nün kenarındaki gece …",0.04


In [8]:
no_city_df = pl.read_parquet("../data/golden_set.parquet").filter(pl.col("geonameIds").list.len() == 0)

In [9]:
results = search_batch(no_city_df["query"].to_list(), top_k=50, use_rerank=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [10]:
results = [r["results"] for r in results]

In [11]:
accuracy = 0
for r in results:
    if r == []:
        accuracy += 1
accuracy /= len(results)

In [12]:
accuracy

0.9272727272727272